# Retry unavailable source versions

This notebook reads the latest state from five crawl journals, selects paper/version pairs marked `unavailable`, and retries the exact source version from arXiv. Results are written to a separate Google Drive folder.

By default, the notebook only checks inputs and makes no network requests. Use only one notebook session for each output folder.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 1. Connect Drive and prepare the runtime

Create `MyDrive/recovery-colab-kit/` on Google Drive and upload `recovery-runtime.zip` there. The package contains the runtime code and five journals; you do not need to upload the full corpus.

Run the cells in order and grant Drive access when Colab prompts you.


In [ ]:
from pathlib import Path
import sys, subprocess, zipfile, hashlib, json

# Upload recovery-runtime.zip to this Drive folder.
KIT = Path("/content/drive/MyDrive/recovery-colab-kit")
ARCHIVE = KIT / "recovery-runtime.zip"
assert ARCHIVE.is_file(), f"Cannot find {ARCHIVE}. Upload recovery-runtime.zip to KIT or change the KIT path."
REPO = Path("/content/recovery-runtime")
REPO.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as bundle:
    for member in bundle.infolist():
        target = (REPO / member.filename).resolve()
        if not target.is_relative_to(REPO.resolve()):
            raise ValueError("Unsafe archive path")
    bundle.extractall(REPO)
manifest = json.loads((REPO / "runtime-manifest.json").read_text())
for name, expected in manifest["sha256"].items():
    with (REPO / name).open("rb") as stream:
        actual = hashlib.file_digest(stream, "sha256").hexdigest()
    assert actual == expected, f"Hash mismatch: {name}"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "requirements-recovery.txt")])
sys.path.insert(0, str(REPO))
print("Runtime and five journals are ready:", REPO)


In [ ]:
from src.benchmark.recovery_review import read_journal, unavailable_by_run, select_s3_sample, review_metadata, retry_unavailable
BASE = REPO / "evidence"
JOURNALS = {
    "only-v1-drive-evidence": BASE / "only-v1-drive-evidence/work/versions.jsonl",
    "multi-dang-0000-4999-drive-evidence": BASE / "multi-dang-0000-4999-drive-evidence/work/versions.jsonl",
    "multi-hieu-0000-4999-drive-evidence": BASE / "multi-hieu-0000-4999-drive-evidence/work/versions.jsonl",
    "multi-thien-0000-4999-drive-evidence": BASE / "multi-thien-0000-4999-drive-evidence/work/versions.jsonl",
    "multi-tien-0000-4999-drive-evidence": BASE / "multi-tien-0000-4999-drive-evidence/work/versions.jsonl",
}
OUTPUT = KIT / "results" / "retry"
RUN_NETWORK = False  # Set to True after the input checks succeed.
assert len(JOURNALS) == 5 and len(set(p.resolve() for p in JOURNALS.values())) == 5
states = {name: read_journal(p) for name,p in JOURNALS.items()}
print("Records per run:", {name:len(s) for name,s in states.items()})
print("Unavailable paper/version pairs:", len(unavailable_by_run(states)))
sample = select_s3_sample(states["only-v1-drive-evidence"].values(), count=1000, seed=42)
print("Papers in S3 sample:", len(sample), "Unique IDs:", len({r["arxiv_id"] for r in sample}))
print("Output folder:", OUTPUT)


In [ ]:
# Set RUN_NETWORK to True in the cell above, then rerun that cell to start.
if RUN_NETWORK:
    summary = retry_unavailable(JOURNALS, OUTPUT)
    print("Status summary:", summary)
else:
    print("Input checks only; no source downloads were requested.")


## 2. Read results and resume after interruption

Results are stored in `KIT/results/retry/`. The `versions.jsonl` file records the status, HTTP response, reason and source journals for each version; `summary.json` contains status counts; `corpus/` contains any successfully downloaded sources.

- `ok`: downloaded with usable TeX; `no_tex`: downloaded without usable TeX.
- `unavailable`: the source endpoint returned HTTP 404 or 410. This alone does not establish that the paper does not exist or that no alternative source is available.
- `inconclusive` and `corrupt`: unresolved; these can be retried on a subsequent run.

If the runtime is interrupted, rerun the cells with the same input package and output folder. Completed records whose files match their hashes, including `unavailable`, are skipped; reusing the same output does not retry them. For an independent new attempt, use a new output folder and preserve the previous evidence.

If arXiv repeatedly blocks requests, the program records an inconclusive result and stops; wait until the block has cleared before resuming. If a journal is damaged by an interrupted write, preserve the output and inspect it before continuing. Review recovered data before integrating it into the original corpus.


## 3. Conclusion from the November 12, 2026 retry

The reviewed run returned **86/86 unavailable versions** across **80 papers**. All 86 records show **HTTP 404** from the source endpoint between **15:33 and 15:37 Vietnam time**; no version was recovered. The five input journals matched the hashes of the evidence used for verification.

This conclusion applies to that run and endpoint. Use the next cell to inspect the current output: another run may produce different results.


In [ ]:
from collections import Counter

journal = OUTPUT / "versions.jsonl"
if journal.exists():
    ket_qua = read_journal(journal)
    print("Papers:", len({paper for paper, version in ket_qua}))
    print("Versions:", len(ket_qua))
    print("Statuses:", dict(Counter(r["status"] for r in ket_qua.values())))
    print("HTTP status codes:", dict(Counter(r.get("http_status") for r in ket_qua.values())))
else:
    print("No result journal exists in the selected output folder.")
